# Mouse DESeq2 team walkthrough

This is a small shared preview of the mouse DE analysis. It is a dummy version of the working notebook: enough to show how the data was organized and what the family-level structure looks like, without exposing the full internal result package.


## What comes from alignment

The DE inputs come from the STAR alignment stage. `ReadsPerGene.out.tab` was parsed into the count matrix, and the alignment/sample summary was turned into the metadata table used for DE families and contrasts.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'differential_expression_all26'
TABLES = DATA / 'tables'
PREVIEWS = DATA / 'previews'


## Subset definition tables

These tables show how the mouse data was split before DE analysis.


In [2]:
design = pd.read_csv(TABLES / 'mouse_de_design_table.tsv', sep='\t')
design.head()


,srr,sample_title,group_label,source_name,platform_family,source_class,geno_class,condition_family,family_id,gc_status,sex,genotype,treatment,include_in_de,excluded_reason
0,SRR30333743,"Control DRG, ipsilateral, replicate 2","Control DRG, ipsilateral",Dorsal root ganglia,NovaSeq X,tissue,control,ipsilateral_sham,family_tissue_sham_novaseqx,PASS,male,Ahr fl/fl,"WT, 1 day post-sham surgery",True,NaN
1,SRR30333744,"Control DRG, ipsilateral, replicate 1","Control DRG, ipsilateral",Dorsal root ganglia,NovaSeq X,tissue,control,ipsilateral_sham,family_tissue_sham_novaseqx,PASS,male,Ahr f/+,"WT, 1 day post-sham surgery",True,NaN
2,SRR30333745,"Control DRG, contralateral, replicate 2","Control DRG, contralateral",Dorsal root ganglia,NovaSeq X,tissue,control,contralateral_sham,family_tissue_sham_novaseqx,PASS,male,Ahr fl/fl,"WT, 1 day post-sham surgery",True,NaN
3,SRR30333746,"Control DRG, contralateral, replicate 1","Control DRG, contralateral",Dorsal root ganglia,NovaSeq X,tissue,control,contralateral_sham,family_tissue_sham_novaseqx,PASS,male,Ahr f/+,"WT, 1 day post-sham surgery",True,NaN
4,SRR30333747,"Control DRG neurons, replicate 3",Control DRG neurons,Dorsal root ganglia neurons,NovaSeq X,neurons,control,neuron_culture,family_neurons_novaseqx,PASS,female,Ahr fl/fl,"WT, 24h post-seeding",True,NaN


The design table shows how samples were assigned into the DE structure used downstream.


In [3]:
family_manifest = pd.read_csv(TABLES / 'family_manifest.tsv', sep='\t')
family_manifest


,family_id,family_label,samples_total,genes_before_filter,genes_after_filter,design
0,family_tissue_novaseq6000,Tissue / NovaSeq 6000 / naive vs injury,12,78334,18893,~condition_family + geno_class + condition_fam...
1,family_tissue_sham_novaseqx,Tissue / NovaSeq X / ipsilateral vs contralate...,8,78334,19498,~condition_family + geno_class + condition_fam...
2,family_neurons_novaseqx,Neurons / NovaSeq X / genotype effect,6,78334,18988,~geno_class


The family manifest lists the three DE families and the design formula used inside each one.


In [4]:
contrast_manifest = pd.read_csv(TABLES / 'contrast_manifest.tsv', sep='\t')
contrast_manifest


,family_id,contrast_id,contrast_label,result_method,n_tested,n_significant,full_table,sig_table,top_table,volcano_png,ma_png,heatmap_png
0,family_tissue_novaseq6000,geno_in_naive,Genotype effect in naive tissue (CKO vs control),coef: geno_class_cko_vs_control,18888,2,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
1,family_tissue_novaseq6000,geno_in_injury,Genotype effect in injury tissue (CKO vs control),list: geno_class_cko_vs_control + condition_fa...,18888,2,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
2,family_tissue_novaseq6000,injury_in_control,Injury effect in control tissue,coef: condition_family_injury_vs_naive,18888,4667,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
3,family_tissue_novaseq6000,injury_in_cko,Injury effect in CKO tissue,list: condition_family_injury_vs_naive + condi...,18888,4088,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
4,family_tissue_novaseq6000,interaction,Interaction term: extra injury effect in CKO t...,coef: condition_familyinjury.geno_classcko,18888,0,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
5,family_tissue_sham_novaseqx,geno_in_contralateral_sham,Genotype effect in contralateral sham tissue (...,coef: geno_class_cko_vs_control,19498,13,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
6,family_tissue_sham_novaseqx,geno_in_ipsilateral_sham,Genotype effect in ipsilateral sham tissue (CK...,list: geno_class_cko_vs_control + condition_fa...,19498,144,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
7,family_tissue_sham_novaseqx,ipsilateral_vs_contralateral_in_control,Ipsilateral vs contralateral sham effect in co...,coef: condition_family_ipsilateral_sham_vs_con...,19498,52,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...
8,family_tissue_sham_novaseqx,ipsilateral_vs_contralateral_in_cko,Ipsilateral vs contralateral sham effect in CK...,list: condition_family_ipsilateral_sham_vs_con...,19498,131,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/pitergarcia/DataScience/Semester5/BIOL5...,/Users/piterg

The contrast manifest shows which comparisons were defined within each family.


## Family sample membership

Each family has its own `sample_table.tsv` showing which samples belong to that subset.


In [5]:
for family in ['family_tissue_novaseq6000', 'family_tissue_sham_novaseqx', 'family_neurons_novaseqx']:
    print(f'\n=== {family} ===')
    display(pd.read_csv(DATA / family / 'tables' / 'sample_table.tsv', sep='\t').head())



=== family_tissue_novaseq6000 ===


,srr,sample_title,geno_class,condition_family,platform_family,gc_status
0,SRR30333757,"Conditional Knockout DRG, injury, replicate 3",cko,injury,NovaSeq 6000,WARN
1,SRR30333758,"Conditional Knockout DRG, injury, replicate 2",cko,injury,NovaSeq 6000,WARN
2,SRR30333759,"Conditional Knockout DRG, injury, replicate 1",cko,injury,NovaSeq 6000,WARN
3,SRR30333760,"Control DRG, injury, replicate 3",control,injury,NovaSeq 6000,WARN
4,SRR30333761,"Control DRG, injury, replicate 2",control,injury,NovaSeq 6000,WARN



=== family_tissue_sham_novaseqx ===


,srr,sample_title,geno_class,condition_family,platform_family,gc_status
0,SRR30333755,"Conditional Knockout DRG, contralateral, repli...",cko,contralateral_sham,NovaSeq X,PASS
1,SRR30333756,"Conditional Knockout DRG, contralateral, repli...",cko,contralateral_sham,NovaSeq X,WARN
2,SRR30333745,"Control DRG, contralateral, replicate 2",control,contralateral_sham,NovaSeq X,PASS
3,SRR30333746,"Control DRG, contralateral, replicate 1",control,contralateral_sham,NovaSeq X,PASS
4,SRR30333753,"Conditional Knockout DRG, ipsilateral, replica...",cko,ipsilateral_sham,NovaSeq X,PASS



=== family_neurons_novaseqx ===


,srr,sample_title,geno_class,condition_family,platform_family,gc_status
0,SRR30333750,"Conditional Knockout DRG neurons, replicate 3",cko,neuron_culture,NovaSeq X,PASS
1,SRR30333751,"Conditional Knockout DRG neurons, replicate 2",cko,neuron_culture,NovaSeq X,PASS
2,SRR30333752,"Conditional Knockout DRG neurons, replicate 1",cko,neuron_culture,NovaSeq X,PASS
3,SRR30333747,"Control DRG neurons, replicate 3",control,neuron_culture,NovaSeq X,PASS
4,SRR30333748,"Control DRG neurons, replicate 2",control,neuron_culture,NovaSeq X,PASS


## Preview plots

These are selected family-level preview plots from the DE workflow. They give a quick look at the structure of each subset without sharing the full downstream result set.


### Tissue / NovaSeq 6000

Main tissue injury family.


In [6]:
from IPython.display import HTML, display
html = f'''
<div style="display:flex; justify-content:center;">
  <div style="display:grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap:16px; width:95%; align-items:start;">
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">PCA</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:center;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_novaseq6000/pca.png" style="max-width:110%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Sample Distance</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:right;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_novaseq6000/sample_distance_heatmap.png" style="max-width:95%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Dispersion</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:left;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_novaseq6000/dispersion.png" style="max-width:97%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div></div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Volcano</div>
      <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_novaseq6000/injury_in_control_volcano.png" style="max-width:120%; max-height:100%; ">
    </div>
    <div style="display:flex; align-items:right; justify-content:right;">
      <div style="border:1px solid #ccc; border-radius:8px; padding:16px; width:100%; max-width:160px;">
        <div style="font-weight:600; margin-bottom:12px; text-align:center;">Gene Counts</div>
        <div style="margin-bottom:8px;"><span style="font-weight:600; color:#b22222;">Significant:</span> 4,667</div>
        <div><span style="font-weight:600; color:#555;">Not significant:</span> 14,221</div>
      </div>
    </div>
  </div>
</div>
'''
display(HTML(html))


From what I have looked at so far, this is the cleanest family and the one with the strongest signal. The clustering and the volcano both suggest that the injury-focused tissue subset is where the clearest biological change is showing up.

I am still looking into what this means for the project next step, but at this stage it already tells us that this family is the strongest place to focus deeper interpretation.


### Tissue sham / NovaSeq X

Supporting sham-side family.


In [7]:
from IPython.display import HTML, display
html = f'''
<div style="display:flex; justify-content:center;">
  <div style="display:grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap:16px; width:95%; align-items:start;">
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">PCA</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:center;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_sham_novaseqx/pca.png" style="max-width:110%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Sample Distance</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:right;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_sham_novaseqx/sample_distance_heatmap.png" style="max-width:95%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Dispersion</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:left;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_sham_novaseqx/dispersion.png" style="max-width:97%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div></div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Volcano</div>
      <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_tissue_sham_novaseqx/ipsilateral_vs_contralateral_in_control_volcano.png" style="max-width:120%; max-height:100%; ">
    </div>
    <div style="display:flex; align-items:right; justify-content:right;">
      <div style="border:1px solid #ccc; border-radius:8px; padding:16px; width:100%; max-width:160px;">
        <div style="font-weight:600; margin-bottom:12px; text-align:center;">Gene Counts</div>
        <div style="margin-bottom:8px;"><span style="font-weight:600; color:#b22222;">Significant:</span> 52</div>
        <div><span style="font-weight:600; color:#555;">Not significant:</span> 19,446</div>
      </div>
    </div>
  </div>
</div>
'''
display(HTML(html))


From what I have looked at so far, this family is usable, but it is not nearly as strong or as clean as the main tissue family. The structure is there, but the contrast signal looks much lighter, so this feels more like supporting context than the main story.

I am still looking into what this means for the project next step, but right now it suggests that sham-side results should probably be treated carefully and used to support, not drive, the interpretation.


### Neurons / NovaSeq X

Supporting neuron family.


In [8]:
from IPython.display import HTML, display
html = f'''
<div style="display:flex; justify-content:center;">
  <div style="display:grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap:16px; width:95%; align-items:start;">
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">PCA</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:center;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_neurons_novaseqx/pca.png" style="max-width:110%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Sample Distance</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:right;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_neurons_novaseqx/sample_distance_heatmap.png" style="max-width:95%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Dispersion</div>
      <div style="height:320px; display:flex; align-items:center; justify-content:left;">
        <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_neurons_novaseqx/dispersion.png" style="max-width:97%; max-height:100%; object-fit:contain;">
      </div>
    </div>
    <div></div>
    <div>
      <div style="font-weight:600; margin-bottom:8px; text-align:center;">Volcano</div>
      <img src="https://raw.githubusercontent.com/pzg8794/mouse_group_project_work/main/data/differential_expression_all26/previews/family_neurons_novaseqx/geno_in_neurons_volcano.png" style="max-width:120%; max-height:100%; ">
    </div>
    <div style="display:flex; align-items:right; justify-content:right;">
      <div style="border:1px solid #ccc; border-radius:8px; padding:16px; width:100%; max-width:160px;">
        <div style="font-weight:600; margin-bottom:12px; text-align:center;">Gene Counts</div>
        <div style="margin-bottom:8px;"><span style="font-weight:600; color:#b22222;">Significant:</span> 139</div>
        <div><span style="font-weight:600; color:#555;">Not significant:</span> 18,846</div>
      </div>
    </div>
  </div>
</div>
'''
display(HTML(html))


From what I have looked at so far, the neuron family shows a modest but still usable signal. It is not as strong as the main tissue family, but it does look like there is enough structure here to justify keeping it in the analysis.

I am still looking into what this means for the project next step, but at this point it helps us understand that the neuron subset can add useful context even if it is not the main result family.
